# Let's find objects, from a photo to a live camera

Shoutout to [D-FINE](https://github.com/Peterande/D-FINE) and the [ExecuTorch object-detection demo](https://executorch.ai/playground/webgpu/#/demo/detect) for the inspiration!

Let's load a photo, find the objects in it, then try our camera or a video. Adjust the confidence slider and compare the original frame with the detections. Your photos and camera frames stay in this browser.

Choose CPU or WebGPU with `DEVICE` and run the cells from top to bottom in a fresh kernel. We're using D-FINE Small's pretrained COCO weights, with 640×640 inputs and FP32 values.


In [ ]:
import base64
import cv2
import copy
import json
import statistics
import time
from pathlib import Path
from IPython.display import display

import numpy as np
import torch
from IPython.display import Image as DisplayImage
import sys
sys.path.insert(0, str(Path('dfine_models.zip').resolve()))

from dfine_models.nn.backbone.hgnetv2 import HGNetv2
from dfine_models.zoo.dfine.dfine import DFINE
from dfine_models.zoo.dfine.dfine_decoder import DFINETransformer
from dfine_models.zoo.dfine.hybrid_encoder import HybridEncoder


DEVICE = 'cpu'
torch.autograd.set_multithreading_enabled(False)
torch.manual_seed(11)
print('PyTorch:', torch.__version__, '| device:', DEVICE)

print('OpenCV:', cv2.__version__)


## 1. Load the pretrained model

The model code and pretrained weights are included alongside this notebook.


In [ ]:
model = DFINE(
    backbone=HGNetv2(name='B0', return_idx=[1, 2, 3], freeze_at=-1,
                    freeze_norm=False, use_lab=True, pretrained=False),
    encoder=HybridEncoder(in_channels=[256, 512, 1024], hidden_dim=256,
                          depth_mult=0.34, expansion=0.5,
                          eval_spatial_size=[640, 640]),
    decoder=DFINETransformer(feat_channels=[256, 256, 256],
                            num_layers=3, num_points=[3, 6, 3],
                            eval_spatial_size=[640, 640]),
).eval()
checkpoint = torch.load('dfine_s_coco.pth', map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['model'], strict=True)
del checkpoint
for parameter in model.parameters():
    parameter.requires_grad_(False)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')


## 2. Pick a photo

Start with the included Labrador photo or change the filename below. We'll fit it inside a 640×640 canvas while keeping its proportions.


In [ ]:
def prepare_frame(frame):
    height, width = frame.shape[:2]
    scale = min(640 / width, 640 / height)
    fitted_width = max(1, round(width * scale))
    fitted_height = max(1, round(height * scale))
    left, top = (640 - fitted_width) // 2, (640 - fitted_height) // 2
    canvas = np.full((640, 640, 3), 114, dtype=np.uint8)
    canvas[top:top + fitted_height, left:left + fitted_width] = cv2.resize(
        frame, (fitted_width, fitted_height), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    tensor = torch.from_numpy(rgb.transpose(2, 0, 1).copy()).unsqueeze(0)
    return tensor, (left, top, width / fitted_width, height / fitted_height)



photo = cv2.imread('YellowLabradorLooking_new.jpg')
assert photo is not None
values, _ = prepare_frame(photo)
canvas = cv2.cvtColor(np.rint(values[0].permute(1, 2, 0).numpy() * 255).astype(np.uint8),
                      cv2.COLOR_RGB2BGR)
ok, encoded = cv2.imencode('.png', canvas)
assert ok
display(DisplayImage(data=encoded.tobytes()))
print('Input:', tuple(values.shape))


## 3. Check the starting prediction

Run the model on CPU once so we can check the compiled predictions against it.


In [ ]:
started = time.perf_counter()
with torch.inference_mode():
    expected = model(values)
eager_seconds = time.perf_counter() - started
print(f'Eager CPU call: {eager_seconds:.3f} s')
print({name: tuple(value.shape) for name, value in expected.items()})


## 4. Compile for the selected device

The first call includes compilation. Later calls reuse the compiled model and keep its weights on the selected device.


In [ ]:
if DEVICE == 'webgpu':
    from iree.runtime.webgpu import get_webgpu_device
    from iree.turbine.runtime.torch_device import TorchDevice
    gpu = TorchDevice(await get_webgpu_device(
        max_concurrent_operations=len(model.state_dict()) + 1), name='webgpu')
    backend = gpu.backend
else:
    import iree.turbine.runtime.torch_device
    gpu = None
    backend = 'turbine_cpu'

device_model = copy.deepcopy(model).to(DEVICE)
device_input = values.to(DEVICE)
if gpu is not None:
    await gpu.synchronize()
compiled_model = torch.compile(device_model, backend=backend,
                               fullgraph=True, dynamic=False)
print('Compiling the model...', flush=True)
started = time.perf_counter()
with torch.inference_mode():
    answer = compiled_model(device_input)
if gpu is not None:
    await gpu.synchronize()
cold_seconds = time.perf_counter() - started
print(f'Cold first call: {cold_seconds:.3f} s')


In [ ]:
outputs = {}
for name in ('pred_logits', 'pred_boxes'):
    outputs[name] = await gpu.to_cpu(answer[name]) if gpu is not None else answer[name].detach()
    torch.testing.assert_close(outputs[name], expected[name], rtol=3e-3, atol=1e-3)
    print(name, 'matches eager PyTorch; max absolute error:',
          float((outputs[name] - expected[name]).abs().max()))


## 5. Draw the detections

Show the boxes and labels above `SCORE_THRESHOLD`. Increase the threshold to show fewer detections, or lower it to see more.


In [ ]:
SCORE_THRESHOLD = 0.35
classes = json.loads(Path('coco_classes.json').read_text())
scores, indices = outputs['pred_logits'][0].sigmoid().flatten().topk(300)
labels = indices % len(classes)
queries = indices // len(classes)
boxes = outputs['pred_boxes'][0][queries]
centres, sizes = boxes[:, :2], boxes[:, 2:]
boxes_xyxy = torch.cat((centres - sizes / 2, centres + sizes / 2), dim=-1) * 640

annotated = canvas.copy()
detections = []
for score, label, box in zip(scores.tolist(), labels.tolist(), boxes_xyxy.tolist()):
    if score < SCORE_THRESHOLD:
        continue
    box = [min(639, max(0, value)) for value in box]
    name = classes[label]
    x1, y1, x2, y2 = [round(value) for value in box]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (80, 220, 70), 2)
    cv2.putText(annotated, f'{name} {score:.0%}', (x1, max(18, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (80, 220, 70), 2, cv2.LINE_AA)
    detections.append(dict(label=name, score=round(score, 4), box_xyxy=[round(x, 1) for x in box]))
cv2.imwrite('dfine-detections.png', annotated)
ok, encoded = cv2.imencode('.png', annotated)
assert ok
display(DisplayImage(data=encoded.tobytes()))
print(json.dumps(detections, indent=2))


## 6. Time the model

Measure three completed predictions after compilation, then time copying the outputs back separately. Run one notebook at a time when comparing CPU and WebGPU.


In [ ]:
times = []
for _ in range(3):
    started = time.perf_counter()
    with torch.inference_mode():
        answer = compiled_model(device_input)
    if gpu is not None:
        await gpu.synchronize()
    times.append(time.perf_counter() - started)
started = time.perf_counter()
if gpu is not None:
    final_outputs = {name: await gpu.to_cpu(value) for name, value in answer.items()}
else:
    final_outputs = {name: value.detach().clone() for name, value in answer.items()}
readback_seconds = time.perf_counter() - started
for name in ('pred_logits', 'pred_boxes'):
    torch.testing.assert_close(final_outputs[name], expected[name], rtol=3e-3, atol=1e-3)
metrics = dict(model='D-FINE Small COCO', device=DEVICE, precision='FP32', input=[1,3,640,640],
               cold_first_call_seconds=cold_seconds, warmed_completed_seconds=times,
               completed_model_median_ms=statistics.median(times)*1000,
               output_copy_or_readback_ms=readback_seconds*1000,
               checks_passed=True)
report = dict(metrics, passed=True)
print(json.dumps(metrics, indent=2))
Path(f'dfine-{DEVICE}-timings.json').write_text(json.dumps(metrics, indent=2))


## 7. Use the same model for each frame

Keep the compiled model ready and draw its detections onto each new frame.


In [ ]:
async def process_frame(jpeg, threshold):
    started = time.perf_counter()
    threshold = float(threshold)
    if not 0 <= threshold <= 1:
        raise ValueError('Confidence must be between zero and one.')
    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None or max(frame.shape[:2]) > 1024:
        raise ValueError('Expected a valid camera image no larger than 1024 pixels.')
    prepared, (left, top, scale_x, scale_y) = prepare_frame(frame)
    prepared_at = time.perf_counter()
    device_input = prepared.to(DEVICE)
    with torch.inference_mode():
        prediction = compiled_model(device_input)
    if gpu is not None:
        actual = {name: await gpu.to_cpu(prediction[name])
                  for name in ('pred_logits', 'pred_boxes')}
    else:
        actual = {name: prediction[name].detach()
                  for name in ('pred_logits', 'pred_boxes')}
    model_at = time.perf_counter()
    if not all(torch.isfinite(value).all() for value in actual.values()):
        raise ValueError('The detector returned non-finite outputs.')

    # D-FINE's postprocessor selects the top 300 query/class scores.
    scores, indices = actual['pred_logits'][0].sigmoid().flatten().topk(300)
    labels, queries = indices % len(classes), indices // len(classes)
    boxes = actual['pred_boxes'][0][queries]
    centres, sizes = boxes[:, :2], boxes[:, 2:]
    boxes = torch.cat((centres - sizes / 2, centres + sizes / 2), dim=-1) * 640
    palette = [(238, 117, 71), (79, 195, 255), (180, 80, 235),
               (90, 210, 85), (210, 140, 40), (100, 75, 240)]
    detections = []
    for score, label, box in zip(scores.tolist(), labels.tolist(), boxes.tolist()):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        box = [max(0, min(frame.shape[1] - 1, (x1 - left) * scale_x)),
               max(0, min(frame.shape[0] - 1, (y1 - top) * scale_y)),
               max(0, min(frame.shape[1] - 1, (x2 - left) * scale_x)),
               max(0, min(frame.shape[0] - 1, (y2 - top) * scale_y))]
        if box[2] <= box[0] or box[3] <= box[1]:
            continue
        colour = palette[label % len(palette)]
        x1, y1, x2, y2 = [round(value) for value in box]
        cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
        caption = f'{classes[label]} {score:.0%}'
        (text_width, text_height), baseline = cv2.getTextSize(
            caption, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        text_x = min(x1, max(0, frame.shape[1] - text_width - 8))
        text_y = max(text_height + 5, y1 - 5)
        cv2.rectangle(frame, (text_x, text_y - text_height - 5),
                      (text_x + text_width + 8, text_y + baseline + 2), colour, -1)
        cv2.putText(frame, caption, (text_x + 4, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (12, 18, 25), 1, cv2.LINE_AA)
        detections.append(dict(label=classes[label], score=round(score, 4),
                               box_xyxy=[round(value, 1) for value in box]))
    postprocessed_at = time.perf_counter()
    ok, encoded = cv2.imencode('.png', frame, [cv2.IMWRITE_PNG_COMPRESSION, 1])
    if not ok:
        raise RuntimeError('OpenCV could not encode the result.')
    png = base64.b64encode(encoded).decode()
    finished = time.perf_counter()
    timings = dict(prepare_ms=1000 * (prepared_at - started),
                   model_ms=1000 * (model_at - prepared_at),
                   postprocess_ms=1000 * (postprocessed_at - model_at),
                   encode_ms=1000 * (finished - postprocessed_at),
                   python_total_ms=1000 * (finished - started))
    return dict(png=png, threshold=threshold, detections=detections, timings=timings)


## 8. Try your camera

Click **Start camera** and allow access, or try the sample photo, upload a photo or upload a video. Adjust **Confidence**, switch the view and drag the before/after slider.

The panel shows completed updates per second, including image processing and display. Click **Stop** to release the camera.


In [ ]:
from live_detection import LiveDetection

if 'camera' in globals():
    await camera.close()
camera = LiveDetection(process_frame, sample='YellowLabradorLooking_new.jpg',
                       size=640, device=DEVICE)


To close the panel from Python, run `await camera.close()`. Re-running its cell closes the previous panel first.

Model and weights: [D-FINE](https://github.com/Peterande/D-FINE). Photo: [Yellow Labrador](https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg), Elf, with derivatives by Djmirko and FT2, CC BY-SA 3.0. Source details are in `sources.json`.
